# Komplettering: Storytelling Charts

This notebook contains only two storytelling charts for the resubmission.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('data/interim/master_cleaned.csv', parse_dates=['date'])
df = df.sort_values(['coin_id', 'date']).copy()

plt.rcParams['figure.facecolor'] = 'white'

In [ ]:
# Story 1: Early meme-coin spikes vs long-term stability.
coins = ['bitcoin', 'official-trump', 'dogwifcoin', 'floki']

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_facecolor('white')
ax.grid(False)

styles = {
    'Bitcoin': {'color': '#D62728', 'lw': 3.0, 'alpha': 1.0},
    'Official Trump': {'color': '#A9AFB7', 'lw': 1.8, 'alpha': 0.55},
    'Dogwifcoin': {'color': '#B8BEC6', 'lw': 1.8, 'alpha': 0.55},
    'Floki': {'color': '#C4CAD1', 'lw': 1.8, 'alpha': 0.55},
}

peak_points = []
end_points = []

for coin in coins:
    d = df[df['coin_id'] == coin].copy()
    d['index_100'] = (d['price'] / d['price'].iloc[0]) * 100
    label = coin.replace('-', ' ').title()
    st = styles[label]

    ax.plot(d['date'], d['index_100'], color=st['color'], lw=st['lw'], alpha=st['alpha'])

    end_points.append((label, d['date'].iloc[-1], d['index_100'].iloc[-1], st['color']))
    if label != 'Bitcoin':
        pidx = d['index_100'].idxmax()
        peak_points.append((label, d.loc[pidx, 'date'], d.loc[pidx, 'index_100']))
    else:
        bitcoin_end = (d['date'].iloc[-1], d['index_100'].iloc[-1])

# Direct labels near line ends (no legend).
for label, x, y, c in end_points:
    ax.annotate(label, xy=(x, y), xytext=(8, 0), textcoords='offset points',
                color=c, fontsize=10, va='center')

# Annotation 1: Meme-coin peak.
top_peak = max(peak_points, key=lambda x: x[2])
ax.annotate(
    'Meme-coin peak',
    xy=(top_peak[1], top_peak[2]),
    xytext=(22, -26),
    textcoords='offset points',
    color='#D62728',
    fontsize=10,
    fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#D62728', lw=1.8)
)

# Annotation 2: Bitcoin stability.
ax.annotate(
    'Bitcoin keeps value better over time',
    xy=(bitcoin_end[0], bitcoin_end[1]),
    xytext=(-230, 22),
    textcoords='offset points',
    color='#1A7F37',
    fontsize=10,
    fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#1A7F37', lw=1.8)
)

ax.set_title('Story 1: Meme Coins Spike Early, Bitcoin Stays More Stable', fontsize=14, pad=12)
ax.set_xlabel('Date')
ax.set_ylabel('Indexed Price (Start = 100)')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#BDBDBD')
ax.spines['bottom'].set_color('#BDBDBD')

plt.tight_layout()
plt.show()


In [ ]:
# Story 2: Simple risk vs return comparison.
coins = ['bitcoin', 'official-trump', 'dogwifcoin', 'floki']
rows = []

for coin in coins:
    d = df[df['coin_id'] == coin].copy()
    d['daily_return'] = d['price'].pct_change()
    total_return = (d['price'].iloc[-1] / d['price'].iloc[0] - 1) * 100
    volatility = d['daily_return'].std() * 100
    rows.append({
        'label': coin.replace('-', ' ').title(),
        'total_return': total_return,
        'volatility': volatility
    })

risk = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_facecolor('white')
ax.grid(False)

colors = {
    'Bitcoin': '#D62728',
    'Official Trump': '#A9AFB7',
    'Dogwifcoin': '#B8BEC6',
    'Floki': '#C4CAD1'
}

for _, r in risk.iterrows():
    size = 130 if r['label'] == 'Bitcoin' else 95
    alpha = 1.0 if r['label'] == 'Bitcoin' else 0.6
    ax.scatter(r['volatility'], r['total_return'], s=size, color=colors[r['label']], alpha=alpha)

    if r['label'] != 'Bitcoin':
        ax.annotate(r['label'], (r['volatility'], r['total_return']),
                    xytext=(7, 6), textcoords='offset points', fontsize=10, color=colors[r['label']])

# Annotate highest risk and best risk-adjusted point (lowest vol among positive returns).
highest_risk = risk.loc[risk['volatility'].idxmax()]
positive = risk[risk['total_return'] > 0]
if not positive.empty:
    stable_winner = positive.loc[positive['volatility'].idxmin()]
else:
    stable_winner = risk.loc[risk['volatility'].idxmin()]

# Add Bitcoin label manually to avoid overlap with the green annotation.
btc = risk[risk['label'] == 'Bitcoin'].iloc[0]
ax.annotate('Bitcoin', (btc['volatility'], btc['total_return']),
            xytext=(8, 10), textcoords='offset points', fontsize=10, color=colors['Bitcoin'])

ax.annotate(
    'Highest risk',
    xy=(highest_risk['volatility'], highest_risk['total_return']),
    xytext=(25, 25),
    textcoords='offset points',
    color='#D62728',
    fontsize=10,
    fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#D62728', lw=1.8)
)

ax.annotate(
    'Most stable among these coins',
    xy=(stable_winner['volatility'], stable_winner['total_return']),
    xytext=(-280, -45),
    textcoords='offset points',
    color='#1A7F37',
    fontsize=10,
    fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#1A7F37', lw=1.8)
)

ax.set_title('Story 2: Risk vs Return (Simple View)', fontsize=14, pad=12)
ax.set_xlabel('Daily Risk (%)')
ax.set_ylabel('Total Return Over Period (%)')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#BDBDBD')
ax.spines['bottom'].set_color('#BDBDBD')

plt.tight_layout()
plt.show()
